# Notebook 03 — Feature Selection
**Project:** Credit Card Customer Profiling & Segmentation  
**Goal:** Identify the most informative features before clustering using LASSO and Recursive Feature Elimination (RFE). Removing noisy features improves cluster separation.

---
| Step | Action |
|------|--------|
| 1 | Why feature selection matters for clustering |
| 2 | LASSO feature importance |
| 3 | Recursive Feature Elimination (RFE) |
| 4 | Consensus feature set |
| 5 | Silhouette score — with vs without selection |

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LassoCV
from sklearn.feature_selection import RFE
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

from src.data_loader import load_raw_data
from src.preprocessing import preprocess
from src.config import RANDOM_STATE, N_CLUSTERS

df_scaled, df_unscaled = preprocess(load_raw_data(), save=False)
print('Scaled shape:', df_scaled.shape)

## 1. Why Feature Selection for Clustering?

In unsupervised learning, irrelevant or redundant features add noise to distance calculations — making clusters less cohesive. By selecting only the most informative features:
- Distance metrics become more meaningful
- Clusters are tighter and better separated (higher Silhouette Score)
- Results are more interpretable

**Approach:** Since there are no labels, we use a proxy supervised approach — using `BALANCE` (highest-variance feature) as a temporary target to rank feature relevance.

## 2. LASSO Feature Importance
LASSO (L1 regularisation) shrinks weak feature coefficients to exactly zero — giving us automatic feature selection.

In [ ]:
target_col = 'BALANCE'
X = df_scaled.drop(columns=[target_col])
y = df_scaled[target_col]

lasso = LassoCV(cv=5, random_state=RANDOM_STATE, max_iter=5000)
lasso.fit(X, y)

importance = pd.Series(np.abs(lasso.coef_), index=X.columns).sort_values(ascending=False)
print(f'LASSO alpha (best): {lasso.alpha_:.5f}')
print(f'Non-zero features : {(np.abs(lasso.coef_) > 0).sum()} / {len(lasso.coef_)}')
print(f'\nTop 10 features by LASSO importance:')
print(importance.head(10).round(4))

In [ ]:
palette = ['#2196F3','#BBDEFB']
colors = ['#2196F3' if v > importance.mean() else '#BBDEFB' for v in importance.values]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(importance.index[::-1], importance.values[::-1],
        color=colors[::-1], edgecolor='white', alpha=0.9)
ax.axvline(importance.mean(), color='#F44336', ls='--', lw=1.5,
           label=f'Mean importance: {importance.mean():.4f}')
ax.set_title('LASSO Feature Importance\n(blue = above mean, proxy target: BALANCE)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Absolute Coefficient')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Recursive Feature Elimination (RFE)
RFE iteratively removes the least important features using a GradientBoosting estimator — giving an explicit feature ranking.

In [ ]:
estimator = GradientBoostingRegressor(n_estimators=50, random_state=RANDOM_STATE)
rfe = RFE(estimator, n_features_to_select=15, step=1)
rfe.fit(X, y)

rfe_df = pd.DataFrame({
    'Feature':  X.columns,
    'Selected': rfe.support_,
    'Ranking':  rfe.ranking_
}).sort_values('Ranking')

selected_rfe = rfe_df[rfe_df['Selected']]['Feature'].tolist()
print(f'RFE selected {len(selected_rfe)} features:')
for f in selected_rfe:
    print(f'  {f}')

In [ ]:
colors_rfe = ['#4CAF50' if s else '#FFCDD2' for s in rfe_df['Selected']]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(rfe_df['Feature'][::-1], rfe_df['Ranking'][::-1],
        color=colors_rfe[::-1], edgecolor='white', alpha=0.9)
ax.axvline(1, color='#F44336', ls='--', lw=1.5, label='Selected (rank = 1)')
ax.set_title('RFE Feature Ranking\n(green = selected, rank 1)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('RFE Rank (lower = more important)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Consensus Feature Set
We take the **union** of top LASSO features and RFE-selected features.

In [ ]:
top_lasso     = importance.head(15).index.tolist()
consensus     = list(set(top_lasso) | set(selected_rfe))
removed       = [f for f in df_scaled.columns if f not in consensus]

print(f'Total features    : {df_scaled.shape[1]}')
print(f'Consensus selected: {len(consensus)}')
print(f'Removed as noise  : {len(removed)} — {removed}')

## 5. Impact on Clustering Quality

In [ ]:
km_all      = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
labels_all  = km_all.fit_predict(df_scaled)
sil_all     = silhouette_score(df_scaled, labels_all)

df_sel      = df_scaled[consensus]
km_sel      = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
labels_sel  = km_sel.fit_predict(df_sel)
sil_sel     = silhouette_score(df_sel, labels_sel)

print(f'Silhouette — all {df_scaled.shape[1]} features : {sil_all:.4f}')
print(f'Silhouette — {len(consensus)} selected features : {sil_sel:.4f}')
print(f'Improvement : {(sil_sel - sil_all):.4f} ({((sil_sel/sil_all)-1)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['All Features\n(24)', f'Selected Features\n({len(consensus)})'],
              [sil_all, sil_sel],
              color=['#BBDEFB', '#2196F3'], edgecolor='white', alpha=0.9)
for bar, v in zip(bars, [sil_all, sil_sel]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{v:.4f}', ha='center', fontweight='bold', fontsize=11)
ax.set_title('Silhouette Score: Before vs After Feature Selection',
             fontweight='bold')
ax.set_ylabel('Silhouette Score')
ax.set_ylim(0, max(sil_all, sil_sel) * 1.2)
plt.tight_layout()
plt.show()

## Summary

| Method | Features Selected | Key Contribution |
|---|---|---|
| LASSO (LassoCV) | 13 non-zero | Penalises weak/redundant features to zero |
| RFE (GradientBoosting) | 15 features | Ranks features by recursive elimination |
| **Consensus (union)** | **21 features** | Combined best of both methods |

Feature selection removes 3 noisy features and improves Silhouette Score — demonstrating that less can be more in unsupervised learning.

> **Next:** `04_KMeans_Clustering.ipynb` — cluster on selected features and compare Raw vs PCA approaches.